# PointNet++ Extrusion Segmentation

In this notebook I will explore how extrusion segmentation can be done using PointNet++.

In [1]:
import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append("..")
sys.path.append("../code")

from dataset import PCExtrusionSegmentationDataset
from models.DeepCAD.cadlib.visualize import vec2CADsolid
from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import create_CAD
from models.DeepCAD.cadlib.visualize import CADsolid2pc
from models.DeepCAD.utils.pc_utils import write_ply
import open3d as o3d
from metrics import ClassificationRunningScore
import torch

In [2]:
def visualize_labeled_pc(points, labels):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    colors = plt.cm.tab10(labels / labels.max())[:, :3]
    pcd.colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([pcd])

In [3]:
train_dataset = PCExtrusionSegmentationDataset("../data", 'train', use_normals=False, verbose=False)
val_dataset = PCExtrusionSegmentationDataset("../data", 'validation', use_normals=False, verbose=False)
test_dataset = PCExtrusionSegmentationDataset("../data", 'test', use_normals=False, verbose=False)
datasets = [train_dataset, val_dataset, test_dataset]

In [4]:
a = set()
a.add("00675619")

In [6]:
from tqdm import tqdm
ids = set()
for dataset in datasets:
    length = len(dataset)
    for i in tqdm(range(length)):
        data = dataset[i]
        pc = data['pc'].numpy()
        label = data['label'].numpy()
        id = data['id']
        
        ids.add(id)
        assert pc.shape[0] == label.shape[0]

100%|██████████████████████████████████████| 8035/8035 [00:16<00:00, 492.00it/s]


In [7]:
len(ids)

177776

In [297]:
for i in range(0,100):
    data = train_dataset[i]
    pc = data['pc'].numpy()
    label = data['label'].numpy()
    id = data['id']
    visualize_labeled_pc(pc, label)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


KeyboardInterrupt: 

TODO:
- see if every data can be accessed
- visualize data
- sanity check

In [14]:
data = train_dataset[0]

In [15]:
label = data['label']

In [23]:
aha = label[:5]
aha

tensor([3, 3, 2, 4, 3])

In [268]:
import torch
data = train_dataset[0]
label = data['label']
label = torch.tensor(2)#label[:5])
def to_categorical(y, num_classes):
    """ 1-hot encodes a tensor """
    new_y = torch.eye(num_classes)[y.cpu().data.numpy(),]
    if (y.is_cuda):
        return new_y.cuda()
    return new_y
lol = to_categorical(label, 10)
print(lol[:5])

tensor([0., 0., 1., 0., 0.])


## Check PointNetFeaturePropagation

In [280]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from time import time
import numpy as np

def index_points(points, idx):
    """

    Input:
        points: input points data, [B, N, C]
        idx: sample index data, [B, S]
    Return:
        new_points:, indexed points data, [B, S, C]
    """
    device = points.device
    B = points.shape[0]
    view_shape = list(idx.shape)
    view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape)
    repeat_shape[0] = 1
    batch_indices = torch.arange(B, dtype=torch.long).to(device).view(view_shape).repeat(repeat_shape)
    new_points = points[batch_indices, idx, :]
    return new_points

def square_distance(src, dst):
    """
    Calculate Euclid distance between each two points.

    src^T * dst = xn * xm + yn * ym + zn * zm；
    sum(src^2, dim=-1) = xn*xn + yn*yn + zn*zn;
    sum(dst^2, dim=-1) = xm*xm + ym*ym + zm*zm;
    dist = (xn - xm)^2 + (yn - ym)^2 + (zn - zm)^2
         = sum(src**2,dim=-1)+sum(dst**2,dim=-1)-2*src^T*dst

    Input:
        src: source points, [B, N, C]
        dst: target points, [B, M, C]
    Output:
        dist: per-point square distance, [B, N, M]
    """
    B, N, _ = src.shape
    _, M, _ = dst.shape
    dist = -2 * torch.matmul(src, dst.permute(0, 2, 1))
    dist += torch.sum(src ** 2, -1).view(B, N, 1)
    dist += torch.sum(dst ** 2, -1).view(B, 1, M)
    return dist

class PointNetFeaturePropagation(nn.Module):
    def __init__(self, in_channel, mlp):
        super(PointNetFeaturePropagation, self).__init__()
        self.mlp_convs = nn.ModuleList()
        self.mlp_bns = nn.ModuleList()
        last_channel = in_channel
        print("MLPs:")
        for out_channel in mlp:
            print(last_channel, "-->", out_channel)
            self.mlp_convs.append(nn.Conv1d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm1d(out_channel))
            last_channel = out_channel

    def forward(self, xyz1, xyz2, points1, points2):
        if xyz1 is not None:
            print(f"xyz1:    {xyz1.shape}     point coordinates before centroid sampling")
        
        if xyz2 is not None:
            print(f"xyz2:    {xyz2.shape}     point coordinates after centroid sampling")
        
        if points1 is not None:
            print(f"points1: {points1.shape}  features before centroid sampling")
        
        if points2 is not None:
            print(f"points2: {points2.shape}  features after centroid sampling / propagated features")
        
        print("First the last two dimensions are permuted.\n")

        """
        Input:
            xyz1: input points position data, [B, C, N]
            xyz2: sampled input points position data, [B, C, S]
            points1: input points data, [B, D, N]
            points2: input points data, [B, D, S]
        Return:
            new_points: upsampled points data, [B, D', N]
        """
        xyz1 = xyz1.permute(0, 2, 1)
        xyz2 = xyz2.permute(0, 2, 1)

        points2 = points2.permute(0, 2, 1)
        B, N, C = xyz1.shape
        _, S, _ = xyz2.shape

        if S == 1:
            interpolated_points = points2.repeat(1, N, 1)
            print(f"\nIf only 1 centroid, repeat its global feature vector until same number as N = {N}: interpolated_points {interpolated_points.shape}")
        else:
            print(f"\nIf the previous layer has more centroid points ({N}), first calculate the squared distance between the input points"
                  f" {xyz1.shape} and {xyz2.shape}.")
            dists = square_distance(xyz1, xyz2)
            print(f"\nNow for each of the {N} input points we have {S} distances to the {S} points of the centroids of the next layer {dists.shape}")
            dists, idx = dists.sort(dim=-1)
            print(f"\nThe {S} distances per {N} points are sorted ascendingly, meaning that the closest point in xyz2 to the point in xyz1 is now the first one.")
            dists, idx = dists[:, :, :3], idx[:, :, :3]  # [B, N, 3]
            print(f"\nThen only the 3 nearest points are retained {dists.shape} along their indices: {idx.shape}")

            dist_recip = 1.0 / (dists + 1e-8)
            print(f"\nTake the reciproke of the distance: {dist_recip.shape}")
            norm = torch.sum(dist_recip, dim=2, keepdim=True)
            print(f"\nAnd sum the reciprokes up of the three nearest points resulting in the norm: {norm.shape}")
            weight = dist_recip / norm
            print(f"\nUsing this we can calculate weights by dividing the reciproke with the norm: {weight.shape}")
            interpolated_points = torch.sum(index_points(points2, idx) * weight.view(B, N, 3, 1), dim=2)
            print(f"\nFinally, we get the features of the 3 nearest neighbors indexed in xyz2: {index_points(points2, idx).shape}"
                 f" and we multiply them with the weights: {weight.view(B, N, 3, 1).shape}. The result is {(index_points(points2, idx) * weight.view(B, N, 3, 1)).shape}")
            print(f"\nNow we sum up the weighted features of the three points: {interpolated_points.shape}")
            

        if points1 is not None:
            points1 = points1.permute(0, 2, 1)
            new_points = torch.cat([points1, interpolated_points], dim=-1)
            print(
                f"\nIf there are features (points1 {points1.shape}) before centroid sampling, "
                f"concatenate them with the repeated global feature vector/the weighted point features"
                f"(interpolated_points {interpolated_points.shape}): new_points {new_points.shape}"
            )

        else:
            new_points = interpolated_points

        new_points = new_points.permute(0, 2, 1)
        for i, conv in enumerate(self.mlp_convs):
            bn = self.mlp_bns[i]
            new_points = F.relu(bn(conv(new_points)))
        print(f"\nThe points are processed by the MLP's {self.mlp_convs}")
        print(f"\nThereby the outputs points now are {new_points.shape}")
        return new_points

In [281]:
l0_xyz = torch.randn(2, 3, 10000)     
l0_points = torch.randn(2, 3, 10000)  

l1_xyz = torch.randn(2, 3, 1024)
l1_points = torch.randn(2, 96, 1024)

l2_xyz = torch.randn(2, 3, 256)
l2_points = torch.randn(2, 256, 256)

l3_xyz = torch.randn(2, 3, 64)
l3_points = torch.randn(2, 512, 64)

l4_xyz = torch.randn(2, 3, 16)
l4_points = torch.randn(2, 1024, 16)

In [282]:
fp4 = PointNetFeaturePropagation(512+512+256+256, [256, 256])
fp3 = PointNetFeaturePropagation(128+128+256, [256, 256])
fp2 = PointNetFeaturePropagation(32+64+256, [256, 128])
fp1 = PointNetFeaturePropagation(128, [128, 128, 128])

MLPs:
1536 --> 256
256 --> 256
MLPs:
512 --> 256
256 --> 256
MLPs:
352 --> 256
256 --> 128
MLPs:
128 --> 128
128 --> 128
128 --> 128


In [283]:
l3_points = fp4(l3_xyz, l4_xyz, l3_points, l4_points)

xyz1:    torch.Size([2, 3, 64])     point coordinates before centroid sampling
xyz2:    torch.Size([2, 3, 16])     point coordinates after centroid sampling
points1: torch.Size([2, 512, 64])  features before centroid sampling
points2: torch.Size([2, 1024, 16])  features after centroid sampling / propagated features
First the last two dimensions are permuted.


If the previous layer has more centroid points (64), first calculate the squared distance between the input points torch.Size([2, 64, 3]) and torch.Size([2, 16, 3]).

Now for each of the 64 input points we have 16 distances to the 16 points of the centroids of the next layer torch.Size([2, 64, 16])

The 16 distances per 64 points are sorted ascendingly, meaning that the closest point in xyz2 to the point in xyz1 is now the first one.

Then only the 3 nearest points are retained torch.Size([2, 64, 3]) along their indices: torch.Size([2, 64, 3])

Take the reciproke of the distance: torch.Size([2, 64, 3])

And sum the reciprokes up 

In [284]:
l2_points = fp3(l2_xyz, l3_xyz, l2_points, l3_points)

xyz1:    torch.Size([2, 3, 256])     point coordinates before centroid sampling
xyz2:    torch.Size([2, 3, 64])     point coordinates after centroid sampling
points1: torch.Size([2, 256, 256])  features before centroid sampling
points2: torch.Size([2, 256, 64])  features after centroid sampling / propagated features
First the last two dimensions are permuted.


If the previous layer has more centroid points (256), first calculate the squared distance between the input points torch.Size([2, 256, 3]) and torch.Size([2, 64, 3]).

Now for each of the 256 input points we have 64 distances to the 64 points of the centroids of the next layer torch.Size([2, 256, 64])

The 64 distances per 256 points are sorted ascendingly, meaning that the closest point in xyz2 to the point in xyz1 is now the first one.

Then only the 3 nearest points are retained torch.Size([2, 256, 3]) along their indices: torch.Size([2, 256, 3])

Take the reciproke of the distance: torch.Size([2, 256, 3])

And sum the recip

In [285]:
l1_points = fp2(l1_xyz, l2_xyz, l1_points, l2_points)

xyz1:    torch.Size([2, 3, 1024])     point coordinates before centroid sampling
xyz2:    torch.Size([2, 3, 256])     point coordinates after centroid sampling
points1: torch.Size([2, 96, 1024])  features before centroid sampling
points2: torch.Size([2, 256, 256])  features after centroid sampling / propagated features
First the last two dimensions are permuted.


If the previous layer has more centroid points (1024), first calculate the squared distance between the input points torch.Size([2, 1024, 3]) and torch.Size([2, 256, 3]).

Now for each of the 1024 input points we have 256 distances to the 256 points of the centroids of the next layer torch.Size([2, 1024, 256])

The 256 distances per 1024 points are sorted ascendingly, meaning that the closest point in xyz2 to the point in xyz1 is now the first one.

Then only the 3 nearest points are retained torch.Size([2, 1024, 3]) along their indices: torch.Size([2, 1024, 3])

Take the reciproke of the distance: torch.Size([2, 1024, 3])

A

In [286]:
l0_points = fp1(l0_xyz, l1_xyz, None, l1_points)

xyz1:    torch.Size([2, 3, 10000])     point coordinates before centroid sampling
xyz2:    torch.Size([2, 3, 1024])     point coordinates after centroid sampling
points2: torch.Size([2, 128, 1024])  features after centroid sampling / propagated features
First the last two dimensions are permuted.


If the previous layer has more centroid points (10000), first calculate the squared distance between the input points torch.Size([2, 10000, 3]) and torch.Size([2, 1024, 3]).

Now for each of the 10000 input points we have 1024 distances to the 1024 points of the centroids of the next layer torch.Size([2, 10000, 1024])

The 1024 distances per 10000 points are sorted ascendingly, meaning that the closest point in xyz2 to the point in xyz1 is now the first one.

Then only the 3 nearest points are retained torch.Size([2, 10000, 3]) along their indices: torch.Size([2, 10000, 3])

Take the reciproke of the distance: torch.Size([2, 10000, 3])

And sum the reciprokes up of the three nearest points r

In [270]:
import torch
data = train_dataset[0]
label = data['label']
label = torch.tensor(2)#label[:5])
def to_categorical(y, num_classes):
    """ 1-hot encodes a tensor """
    new_y = torch.eye(num_classes)[y.cpu().data.numpy(),]
    if (y.is_cuda):
        return new_y.cuda()
    return new_y
cls_label = to_categorical(label, 10)
print(lol[:5])

tensor([0., 0., 1., 0., 0.])


In [272]:
cls_label_one_hot = cls_label.view(2,16,1).repeat(1,1,10000)

RuntimeError: shape '[2, 16, 1]' is invalid for input of size 10

In [273]:
cls_label.shape

torch.Size([10])

In [287]:
seg_pred = torch.rand((2,10000,10))
seg_pred = seg_pred.contiguous().view(-1, 10)
seg_pred.shape

torch.Size([20000, 10])

In [292]:
label = torch.rand((2, 10000))

In [296]:
label.view(-1,1).squeeze().shape

torch.Size([20000])

## Metrics

In [2]:
score_tracker = ClassificationRunningScore(10)

In [3]:
target = torch.tensor([0,1,2,3,4,5,6,7,8,9])
pred   = torch.tensor([0,0,0,2,2,2,4,4,4,9])

In [4]:
score_tracker.update(pred, target)

In [5]:
scores = score_tracker.get_scores()

In [6]:
scores

{'tp': array([1, 0, 0, 0, 0, 0, 0, 0, 0, 1]),
 'fp': array([2, 0, 3, 0, 3, 0, 0, 0, 0, 0]),
 'fn': array([0, 1, 1, 1, 1, 1, 1, 1, 1, 0])}

In [7]:
score_tracker.get_accuracy()

0.1999999998

In [8]:
score_tracker.get_class_accuracy()

array([0.99999999, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.99999999])

In [9]:
score_tracker.get_mean_class_accuracy()

0.19999999800000004

In [ ]:
score_tracker.reset()